# ☕ Capstone Data Analysis: Coffee Shop Sales

## Tahap 7: Business Recommendation

---

# 1. Objective

Notebook ini bertujuan untuk:

- **Menerjemahkan hasil analisis menjadi tindakan bisnis** yang konkret dan dapat dijalankan
- **Menentukan prioritas tindakan** berdasarkan dampak dan urgensi
- **Menghubungkan temuan, bukti, rekomendasi, dan dampak** dalam satu kerangka kerja yang koheren
- **Menghasilkan minimal 10 rekomendasi bisnis** yang realistis, berbasis data, dan dapat ditindaklanjuti oleh manajemen

Rekomendasi disusun berdasarkan:
- Hasil EDA dari `05_exploratory_data_analysis.ipynb`
- Dashboard interaktif pada `app.py` dan folder `pages/`
- File ringkasan pada `processed/eda/`
- Dataset features dari `processed/coffee_shop_sales_featured.csv`

**Catatan Penting:**
Dataset ini **tidak menyediakan data cost, profit, atau margin**. Seluruh analisis dan rekomendasi difokuskan pada **revenue, transaction volume, quantity, dan average transaction value** sebagai proxy kinerja bisnis.

---

# 2. Business Questions

Notebook ini menjawab pertanyaan-pertanyaan bisnis berikut:

| No | Business Question | Section |
|---|---|---|
| 1 | Tindakan apa yang dapat meningkatkan penjualan? | 6.1 Sales Recommendation |
| 2 | Produk apa yang perlu diprioritaskan? | 6.2 Product Recommendation |
| 3 | Produk apa yang perlu dievaluasi? | 6.2 Product Recommendation |
| 4 | Kapan waktu terbaik menjalankan promosi? | 6.5 Time Recommendation |
| 5 | Bagaimana strategi staffing berdasarkan peak hour? | 6.5 Time Recommendation |
| 6 | Cabang atau wilayah mana yang perlu dikembangkan? | 6.4 Region and Store Recommendation |
| 7 | Segmen pelanggan mana yang perlu dipertahankan? | 6.3 Customer Recommendation |
| 8 | Bagaimana meningkatkan average transaction value? | 6.1 & 6.2 Sales/Product |
| 9 | Bagaimana mengoptimalkan inventory? | 6.2 & 6.5 Product/Time |
| 10 | Apa tiga rekomendasi paling penting untuk manajemen? | 8. Top 3 Priority Recommendations |

---

In [ ]:
# ============================================================
# Import Library
# ============================================================
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Library berhasil diimport.')

---
# 3. Load Supporting Data

Memuat data hasil EDA dan Feature Engineering dari berbagai sumber.

In [ ]:
# ============================================================
# Load Supporting Data
# ============================================================

# 1. Load Featured Dataset
FEATURED_PATH = 'processed/coffee_shop_sales_featured.csv'
df = pd.read_csv(FEATURED_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 2. Load EDA Summary Files
EDA_DIR = 'processed/eda/'

product_summary = pd.read_csv(os.path.join(EDA_DIR, 'product_summary.csv'))
store_summary = pd.read_csv(os.path.join(EDA_DIR, 'store_summary.csv'))
customer_summary = pd.read_csv(os.path.join(EDA_DIR, 'customer_summary.csv'))
hourly_summary = pd.read_csv(os.path.join(EDA_DIR, 'hourly_summary.csv'))
monthly_sales = pd.read_csv(os.path.join(EDA_DIR, 'monthly_sales.csv'))
day_of_week = pd.read_csv(os.path.join(EDA_DIR, 'day_of_week_pattern.csv'))
weekday_weekend = pd.read_csv(os.path.join(EDA_DIR, 'weekday_weekend_summary.csv'))
category_contribution = pd.read_csv(os.path.join(EDA_DIR, 'category_contribution.csv'))
age_group = pd.read_csv(os.path.join(EDA_DIR, 'age_group_analysis.csv'))
city_performance = pd.read_csv(os.path.join(EDA_DIR, 'city_performance.csv'))
country_performance = pd.read_csv(os.path.join(EDA_DIR, 'country_performance.csv'))
store_type = pd.read_csv(os.path.join(EDA_DIR, 'store_type_performance.csv'))
customer_segment = pd.read_csv(os.path.join(EDA_DIR, 'customer_segment_analysis.csv'))

# 3. Calculate Key Metrics from Featured Dataset
total_revenue = df['total_amount'].sum()
total_transactions = df['transaction_id'].nunique()
total_quantity = int(df['quantity'].sum())
avg_transaction_value = df['total_amount'].mean()
n_customers = df['customer_id'].nunique()
n_stores = df['store_id'].nunique()
n_products = df['product_name'].nunique()

print('=' * 70)
print(' SUPPORTING DATA LOADED SUCCESSFULLY')
print('=' * 70)
print(f'\nFile loaded:')
print(f'  - Featured Dataset: {len(df):,} rows x {df.shape[1]} columns')
print(f'  - EDA files: {len(os.listdir(EDA_DIR))} files in {EDA_DIR}')
print(f'\nKey Metrics:')
print(f'  Total Revenue:        ${total_revenue:,.2f}')
print(f'  Total Transactions:   {total_transactions:,}')
print(f'  Total Quantity:       {total_quantity:,}')
print(f'  Avg Transaction Value: ${avg_transaction_value:,.2f}')
print(f'  Unique Customers:     {n_customers:,}')
print(f'  Unique Stores:        {n_stores}')
print(f'  Unique Products:      {n_products}')

---
# 4. Summary of Key Findings

Berikut ringkasan minimal 10 temuan utama dari EDA dan dashboard.

In [ ]:
# ============================================================
# Hitung Metrik Tambahan untuk Key Findings
# ============================================================

# Discount analysis (using discount_impact since discount_applied is always True)
disc_applied = df[df['discount_impact'] != 'No Discount']
disc_not_applied = df[df['discount_impact'] == 'No Discount']
pct_discount_txn = len(disc_applied) / len(df) * 100

disc_rev = disc_applied['total_amount'].sum()
no_disc_rev = disc_not_applied['total_amount'].sum()
disc_avg_txn = disc_applied['total_amount'].mean()
no_disc_avg_txn = disc_not_applied['total_amount'].mean()

# Loyalty analysis
loyalty_members = df[df['loyalty_member'] == True]
pct_loyalty = len(loyalty_members) / len(df) * 100
loyalty_rev = loyalty_members['total_amount'].sum()
non_loyalty_rev = df[df['loyalty_member'] == False]['total_amount'].sum()

# Repeat customer analysis
cust_txn_count = df.groupby('customer_id')['transaction_id'].nunique()
repeat_customers = (cust_txn_count > 1).sum()
repeat_rate = repeat_customers / n_customers * 100
one_time_customers = (cust_txn_count == 1).sum()

# Top & Bottom products
top5_products_rev = product_summary.nlargest(5, 'total_revenue')[['product_name', 'total_revenue', 'total_quantity', 'transaction_count']]
bot5_products_rev = product_summary.nsmallest(5, 'total_revenue')[['product_name', 'total_revenue', 'total_quantity', 'transaction_count']]

# Store top & bottom
top5_stores = store_summary.nlargest(5, 'revenue')
bot5_stores = store_summary.nsmallest(5, 'revenue')

# Peak hours
peak_hours = hourly_summary.nlargest(3, 'transactions')[['hour', 'transactions', 'revenue']]
off_peak_hours = hourly_summary.nsmallest(3, 'transactions')[['hour', 'transactions', 'revenue']]

# Weekend vs Weekday per day
weekday_data = weekday_weekend[weekday_weekend['label'] == 'Weekday']
weekend_data = weekday_weekend[weekday_weekend['label'] == 'Weekend']
avg_weekday_rev_per_day = weekday_data['revenue'].values[0] / 5
avg_weekend_rev_per_day = weekend_data['revenue'].values[0] / 2
weekend_lift_pct = (avg_weekend_rev_per_day - avg_weekday_rev_per_day) / avg_weekday_rev_per_day * 100

# Monthly peak and trough
peak_month = monthly_sales.loc[monthly_sales['revenue'].idxmax()]
trough_month = monthly_sales.loc[monthly_sales['revenue'].idxmin()]
monthly_gap = peak_month['revenue'] - trough_month['revenue']

# Country spread
country_rev_range = country_performance['revenue'].max() - country_performance['revenue'].min()

print('Metrik pendukung berhasil dihitung.')

In [ ]:
# ============================================================
# Tabel Key Findings
# ============================================================

findings = pd.DataFrame([
    {
        'No': 1,
        'Area': 'Sales',
        'Finding': f'Total revenue mencapai ${total_revenue:,.2f} dari {total_transactions:,} transaksi dengan {total_quantity:,} unit terjual',
        'Evidence': f'Average Transaction Value ${avg_transaction_value:,.2f}; median transaksi lebih rendah dari mean (distribusi right-skewed). Rata-rata quantity per transaksi {total_quantity/total_transactions:.2f} unit',
        'Business Meaning': 'Volume transaksi tinggi tetapi average transaction value relatif rendah menunjukkan peluang besar untuk meningkatkan basket size melalui upselling dan bundling'
    },
    {
        'No': 2,
        'Area': 'Product',
        'Finding': f'Kategori Coffee menyumbang 42.36% dari total revenue (${category_contribution.iloc[0]["revenue"]:,.2f}), diikuti Tea 21.62% (${category_contribution.iloc[1]["revenue"]:,.2f})',
        'Evidence': f'Top 2 kategori (Coffee + Tea) menguasai 63.98% revenue. Kategori terlemah: Smoothie 6.93% (${category_contribution.iloc[5]["revenue"]:,.2f})',
        'Business Meaning': 'Konsentrasi revenue tinggi pada 2 kategori utama; mempertahankan kualitas Coffee dan Tea sangat kritis, sementara kategori lain perlu strategi penguatan'
    },
    {
        'No': 3,
        'Area': 'Product',
        'Finding': f'Tote Bar menghasilkan revenue tertinggi (${product_summary.loc[product_summary["rev_rank"]==1, "total_revenue"].values[0]:,.2f}) meskipun hanya 484 unit terjual (peringkat 37 quantity)',
        'Evidence': f'Tote Bag memiliki avg unit price ${product_summary.loc[product_summary["rev_rank"]==1, "avg_unit_price"].values[0]:.2f}, tertinggi kedua setelah merchandise lainnya. Product ini merchandise, bukan F&B',
        'Business Meaning': 'Merchandise (Tote Bag, Coffee Mug) memiliki margin revenue per unit yang sangat tinggi; potensi untuk meningkatkan penjualan merchandise sebagai value-add'
    },
    {
        'No': 4,
        'Area': 'Time',
        'Finding': f'Weekend menghasilkan revenue ${weekend_data["revenue"].values[0]:,.2f} (36.69% total) dari hanya 2 hari, sementara weekday ${weekday_data["revenue"].values[0]:,.2f} (63.31%) dari 5 hari',
        'Evidence': f'Rata-rata revenue harian weekend ${avg_weekend_rev_per_day:,.2f} vs weekday ${avg_weekday_rev_per_day:,.2f}, selisih +{weekend_lift_pct:.1f}%. Saturday adalah hari terbaik ($26,675)',
        'Business Meaning': 'Revenue per hari weekend jauh lebih tinggi dari weekday; optimasi stok, staf, dan promosi harus memaksimalkan potensi weekend sekaligus mengangkat weekday'
    },
    {
        'No': 5,
        'Area': 'Time',
        'Finding': f'Jam sibuk (peak hour): 7:00-10:00 menyumbang {hourly_summary[hourly_summary["hour"].isin([7,8,9])]["transactions"].sum():,} transaksi dari total {total_transactions:,} transaksi',
        'Evidence': f'Jam 8:00 adalah peak hour dengan {int(hourly_summary.loc[hourly_summary["hour"]==8, "transactions"].values[0]):,} transaksi dan ${hourly_summary.loc[hourly_summary["hour"]==8, "revenue"].values[0]:,.2f} revenue. Jam 0:00-5:00 hanya rata-rata ~224 transaksi/jam',
        'Business Meaning': 'Konsentrasi transaksi di pagi hari sangat tinggi; memastikan staf dan inventori optimal di jam 7-10 adalah prioritas operasional'
    },
    {
        'No': 6,
        'Area': 'Region/Store',
        'Finding': f'Airport stores memiliki avg transaction tertinggi (${store_type[store_type["store_type"]=="Airport"]["avg_transaction"].values[0]:.2f}) meskipun jumlah toko paling sedikit (4 dari 45)',
        'Evidence': f'Airport: ${store_type[store_type["store_type"]=="Airport"]["revenue"].values[0]:,.2f} revenue, {int(store_type[store_type["store_type"]=="Airport"]["transactions"].values[0]):,} transaksi. Standalone: ${store_type[store_type["store_type"]=="Standalone"]["revenue"].values[0]:,.2f} dari 28 toko',
        'Business Meaning': 'Format Airport sangat efisien dengan ATV tinggi; strategi ekspansi lebih lanjut ke airport layak dipertimbangkan'
    },
    {
        'No': 7,
        'Area': 'Region/Store',
        'Finding': f'Revenue gap antar negara mencapai ${country_rev_range:,.2f} (USA $47,694.86 vs UK $25,772.96)',
        'Evidence': f'USA memimpin dengan 34.81% dari total revenue. UK di posisi terendah dengan 18.81%. Gap USA-UK: ${country_rev_range:,.2f}',
        'Business Meaning': 'Perlu investigasi penyebab underperformance UK (Manchester, London) dan identifikasi best practice dari USA dan AUS'
    },
    {
        'No': 8,
        'Area': 'Customer',
        'Finding': f'High Value customers menyumbang {customer_segment[customer_segment["customer_segment"]=="High Value"]["rev_pct"].values[0]:.2f}% revenue dari total, meskipun proporsi populasi terbatas',
        'Evidence': f'High Value: {int(customer_segment[customer_segment["customer_segment"]=="High Value"]["n_customers"].values[0]):,} customers, avg transaction ${customer_segment[customer_segment["customer_segment"]=="High Value"]["avg_transaction"].values[0]:.2f}. Low Value: avg transaction $2.91',
        'Business Meaning': 'High value customers adalah aset utama; retensi dan peningkatan frekuensi mereka sangat kritis untuk pertumbuhan revenue'
    },
    {
        'No': 9,
        'Area': 'Customer',
        'Finding': f'Repeat customer rate hanya {repeat_rate:.2f}% ({repeat_customers:,} dari {n_customers:,} customers melakukan lebih dari 1 transaksi)',
        'Evidence': f'{one_time_customers:,} customers ({one_time_customers/n_customers*100:.2f}%) hanya melakukan 1 transaksi. Customer segment High Value didominasi "One-time (1)" transaksi',
        'Business Meaning': 'Retensi pelanggan sangat rendah; investasi dalam program loyalitas dan engagement untuk mengubah one-time menjadi repeat customer prioritas tinggi'
    },
    {
        'No': 10,
        'Area': 'Discount',
        'Finding': f'{pct_discount_txn:.2f}% transaksi memiliki diskon aktif ({len(disc_applied):,} dari {total_transactions:,} transaksi, berdasarkan discount_impact)',
        'Evidence': f'Transaksi diskon: avg ${disc_avg_txn:.2f}. Transaksi non-diskon: avg ${no_disc_avg_txn:.2f}. Selisih ATV: ${abs(disc_avg_txn - no_disc_avg_txn):.2f}',
        'Business Meaning': 'Penggunaan diskon sudah signifikan; perlu evaluasi apakah diskon efektif meningkatkan volume atau justru mengikis revenue. Penggunaan diskon harus lebih terarah'
    },
    {
        'No': 11,
        'Area': 'Product',
        'Finding': f'Medium Cappuccino adalah produk dengan quantity tertinggi ({int(product_summary.nlargest(1, "total_quantity")["total_quantity"].values[0]):,} unit) dan Croissant memiliki transaksi tertinggi ({int(product_summary.nlargest(1, "transaction_count")["transaction_count"].values[0]):,} transaksi)',
        'Evidence': f'Medium Cappuccino: ${product_summary.nlargest(1, "total_quantity")["total_revenue"].values[0]:,.2f} revenue, avg price $3.90. Croissant: ${product_summary.nlargest(1, "transaction_count")["total_revenue"].values[0]:,.2f} revenue, avg price $2.42',
        'Business Meaning': 'Produk high-volume dengan price rendah menjadi traffic driver; menjaga ketersediaan dan kualitasnya sangat penting untuk menarik pelanggan'
    },
    {
        'No': 12,
        'Area': 'Sales',
        'Finding': f'Monthly revenue gap antara bulan tertinggi ({peak_month["month_name"]} ${peak_month["revenue"]:,.2f}) dan terendah ({trough_month["month_name"]} ${trough_month["revenue"]:,.2f}) adalah ${monthly_gap:,.2f}',
        'Evidence': f'Januari dan Desember adalah peak months. Maret adalah trough. Revenue gap: ${monthly_gap:,.2f} ({monthly_gap/trough_month["revenue"]*100:.1f}% dari bulan terendah)',
        'Business Meaning': 'Pola musiman terlihat jelas; strategi promosi dan inventory perlu menyesuaikan untuk mengoptimalkan bulan-bulan rendah'
    },
    {
        'No': 13,
        'Area': 'Region/Store',
        'Finding': f'Store terendah (Manchester Standalone #21: ${bot5_stores.iloc[0]["revenue"]:,.2f}) memiliki revenue {top5_stores.iloc[0]["revenue"]/bot5_stores.iloc[0]["revenue"]:.1f}x lipat lebih rendah dari store terbaik (Los Angeles Airport #8: ${top5_stores.iloc[0]["revenue"]:,.2f})',
        'Evidence': f'Top store: LA Airport ($3,982.38, avg txn $8.70). Bottom store: Manchester #21 ($2,076.92, avg txn $5.28). Gap: ${top5_stores.iloc[0]["revenue"]-bot5_stores.iloc[0]["revenue"]:,.2f}',
        'Business Meaning': 'Performa antar store sangat bervariasi; perlu investigasi root cause perbedaan (lokasi, operasional, demografi) dan best practice sharing'
    },
    {
        'No': 14,
        'Area': 'Customer',
        'Finding': f'Age group 25-34 adalah segmen revenue terbesar (${age_group.iloc[0]["revenue"]:,.2f}, {age_group.iloc[0]["rev_pct"]:.2f}% dari total)',
        'Evidence': f'25-34: {int(age_group.iloc[0]["transactions"]):,} transaksi, avg txn ${age_group.iloc[0]["avg_transaction"]:.2f}. Disusul 35-44: ${age_group.iloc[1]["revenue"]:,.2f} ({age_group.iloc[1]["rev_pct"]:.2f}%)',
        'Business Meaning': 'Demografis muda profesional adalah core customer; strategi marketing dan product development harus mempertimbangkan preferensi segmen ini'
    },
    {
        'No': 15,
        'Area': 'Performance',
        'Finding': f'Seluruh transaksi ({pct_loyalty:.2f}%) dilakukan oleh loyalty members, namun repeat rate masih rendah ({repeat_rate:.2f}%)',
        'Evidence': f'100% transaksi dari loyalty members (${loyalty_rev:,.2f}). Namun hanya {repeat_customers:,} dari {n_customers:,} customers melakukan repeat purchase. Program loyalitas sudah berjalan namun belum optimal dalam mendorong retensi.',
        'Business Meaning': 'Program loyalitas perlu diperkuat: dari sekedar keanggotaan menjadi program yang benar-benar mendorong repeat purchase dan customer lifetime value'
    }
])

print('=' * 120)
print(' SUMMARY OF KEY FINDINGS')
print('=' * 120)
for _, row in findings.iterrows():
    print(f'\n--- Finding #{int(row["No"])}: {row["Area"]} ---')
    print(f'Finding:    {row["Finding"]}')
    print(f'Evidence:   {row["Evidence"]}')
    print(f'Business:   {row["Business Meaning"]}')

---
# 5. Recommendation Framework

Setiap rekomendasi disusun dengan kerangka berikut:

| Komponen | Deskripsi |
|---|---|
| **Problem** | Masalah bisnis yang ingin diatasi |
| **Evidence** | Bukti data yang mendukung |
| **Recommendation** | Tindakan spesifik yang direkomendasikan |
| **Owner** | Pihak yang bertanggung jawab |
| **Timeline** | Quick Win (0-30 hari), Mid Term (1-3 bulan), atau Strategic (3-12 bulan) |
| **Expected Impact** | Dampak yang diharapkan |
| **Priority** | High, Medium, atau Low |
| **Risk** | Risiko potensial |
| **KPI Monitoring** | Metrik yang harus dipantau |

---

---
# 6. Business Recommendations

## 6.1 Sales Recommendations

In [ ]:
# ============================================================
# Sales Recommendations
# ============================================================

sales_recommendations = pd.DataFrame([
    {
        'No': 'R1',
        'Problem': 'Average Transaction Value rendah ($6.85) dari 20,000 transaksi; banyak transaksi single-item',
        'Evidence': f'ATV ${avg_transaction_value:.2f}, median lebih rendah dari mean. Rata-rata quantity per transaksi {total_quantity/total_transactions:.2f} unit. Banyak produk dengan unit price di bawah $5',
        'Recommendation': 'Implementasi strategi bundling dan upselling: "Add a pastry for $2" di POS, bundle Coffee + Pastry dengan diskon 10%, minimum purchase reward',
        'Owner': 'Head of Sales & Marketing',
        'Timeline': 'Quick Win (0-30 hari)',
        'Expected Impact': 'Peningkatan ATV sebesar 5-10% dari baseline $6.85 ke target $7.20-$7.50. Jika berhasil, potensi revenue tambahan dari 20,000 transaksi bisa mencapai $7,000-$13,000 per tahun',
        'Priority': 'High',
        'Risk': 'Cannibalization produk high-margin; pelanggan merasa terlalu dijual terus-menerus',
        'KPI': 'Average Transaction Value, Items per Transaction, Revenue per Transaction'
    },
    {
        'No': 'R2',
        'Problem': f'Revenue gap antar bulan mencapai ${monthly_gap:,.2f} (Januari ${peak_month["revenue"]:,.2f} vs Maret ${trough_month["revenue"]:,.2f})',
        'Evidence': f'Bulan terendah: Maret (${trough_month["revenue"]:,.2f}), bulan tertinggi: Januari (${peak_month["revenue"]:,.2f}). Gap: ${monthly_gap:,.2f} ({monthly_gap/trough_month["revenue"]*100:.1f}%)',
        'Recommendation': 'Jalankan kampanye seasonal promotion di bulan-bulan rendah (Maret, April, Juni, Agustus). Tema: "Mid-Season Special" dengan bundle eksklusif atau limited-time product. Koordinasikan dengan inventory planning.',
        'Owner': 'Marketing Manager',
        'Timeline': 'Mid Term (1-3 bulan)',
        'Expected Impact': 'Mengurangi revenue gap bulanan sebesar 15-20%; menaikkan revenue bulan rendah mendekati rata-rata ~$11,417',
        'Priority': 'Medium',
        'Risk': 'Promosi berlebihan dapat mengurangi margin; perlu A/B testing untuk menentukan format promosi terbaik',
        'KPI': 'Monthly Revenue Gap, Revenue per Month vs Baseline, Campaign ROI'
    },
    {
        'No': 'R3',
        'Problem': f'{pct_discount_txn:.2f}% transaksi menggunakan diskon, namun dampaknya terhadap volume vs revenue belum dioptimasi',
        'Evidence': f'Transaksi diskon: avg ${disc_avg_txn:.2f}. Non-diskon: avg ${no_disc_avg_txn:.2f}. Dataset tidak memiliki data profit, sehingga evaluasi diskon harus berbasis revenue dan volume',
        'Recommendation': 'Racionalisasi penggunaan diskon: fokus diskon pada produk high-volume dengan margin kuat, batasi diskon pada produk already-low-price. Implementasi tiered discount (minimum purchase untuk diskon). Evaluasi diskon secara berkala berdasarkan peningkatan volume transaksi.',
        'Owner': 'Pricing & Revenue Manager',
        'Timeline': 'Mid Term (1-3 bulan)',
        'Expected Impact': 'Peningkatan revenue per transaksi diskon sebesar 3-5% atau stabilisasi volume transaksi tanpa mengorbankan revenue',
        'Priority': 'Medium',
        'Risk': 'Penurunan volume transaksi jika diskon terlalu dibatasi; pelanggan sensitif harga meninggalkan kompetitor',
        'KPI': 'Discount Transaction Rate, Avg Transaction Value (Discount vs Non-Discount), Revenue per Discounted Transaction'
    },
    {
        'No': 'R4',
        'Problem': 'Payment method belum dioptimasi untuk meningkatkan kecepatan transaksi dan customer experience',
        'Evidence': f'Terdapat 4 payment method: Credit Card, Debit Card, Mobile Wallet, Cash. Transaksi cash membutuhkan waktu lebih lama dan berisiko error',
        'Recommendation': 'Dorong penggunaan digital payment (Mobile Wallet) melalui insentif kecil (bonus loyalty points). Sediakan contactless payment di semua store. Kurangi ketergantungan cash.',
        'Owner': 'Operations Manager',
        'Timeline': 'Mid Term (1-3 bulan)',
        'Expected Impact': 'Peningkatan throughput transaksi 5-10%, pengurangan waktu antrian di peak hour',
        'Priority': 'Low',
        'Risk': 'Resistance dari pelanggan yang masih prefer cash; infrastruktur teknologi belum merata di semua store',
        'KPI': 'Payment Method Distribution, Average Transaction Time, Cash vs Digital Ratio'
    }
])

print('Sales Recommendations (4 rekomendasi):')
for _, row in sales_recommendations.iterrows():
    print(f'\n[{row["No"]}] {row["Recommendation"][:100]}...')
    print(f'    Priority: {row["Priority"]} | Timeline: {row["Timeline"]}')

## 6.2 Product Recommendations

In [ ]:
# ============================================================
# Product Recommendations
# ============================================================

product_recommendations = pd.DataFrame([
    {
        'No': 'R5',
        'Problem': 'Merchandise (Tote Bag, Coffee Mug) memiliki revenue per unit tertinggi namun volume rendah (484 dan 456 unit)',
        'Evidence': f'Tote Bag: $7,021.45 revenue, 484 unit, avg $14.65/unit. Coffee Mug: $5,197.39 revenue, 456 unit, avg $11.67/unit. Keduanya di peringkat 36-37 quantity',
        'Recommendation': 'Jadikan merchandise sebagai add-on item di checkout: "Add Tote Bag $14.65" atau "Coffee Mug $11.67 with any coffee purchase". Tampilkan di POS dan area kasir. Buat limited edition seasonal merchandise.',
        'Owner': 'Product Manager & Visual Merchandising',
        'Timeline': 'Quick Win (0-30 hari)',
        'Expected Impact': 'Peningkatan volume merchandise 20-30% dari baseline 484 unit, potensi tambahan revenue $1,400-$2,100 dari Tote Bag saja',
        'Priority': 'High',
        'Risk': 'Overstock merchandise jika permintaan tidak naik; perlu monitor stok dan reorder point',
        'KPI': 'Merchandise Quantity Sold, Revenue per Merchandise Category, Add-on Conversion Rate'
    },
    {
        'No': 'R6',
        'Problem': f'Produk berkinerja rendah (bottom 5: Large Iced Coffee $958, Medium Pumpkin Spice Latte $956, Medium Iced Coffee $795) perlu dievaluasi',
        'Evidence': f'Bottom 5 products total revenue: ${bot5_products_rev["total_revenue"].sum():,.2f} dari ${total_revenue:,.2f} total ({bot5_products_rev["total_revenue"].sum()/total_revenue*100:.2f}%). Quantity rendah: 129-248 unit per produk',
        'Recommendation': 'Evaluasi 5 produk terbawah: (1) Apakah seasonal (Pumpkin Spice)? (2) Apakah perlu reposisi atau rebranding? (3) Apakah perlu discontinu? Pertahankan produk seasonal, pertimbangkan discontinu produk consistently low performer',
        'Owner': 'Product Manager',
        'Timeline': 'Mid Term (1-3 bulan)',
        'Expected Impact': 'Pembebasan inventory space dan fokus sumber daya pada produk berkinerja lebih baik; peningkatan overall product mix efficiency',
        'Priority': 'Medium',
        'Risk': 'Kehilangan niche customers yang menyukai produk tersebut; seasonal product mungkin memiliki value strategic',
        'KPI': 'Revenue per Product, Product Discontinuation Rate, Product Mix Efficiency'
    },
    {
        'No': 'R7',
        'Problem': 'Category Coffee mendominasi 42.36% revenue; ketergantungan berlebihan pada satu kategori',
        'Evidence': f'Coffee: ${category_contribution.iloc[0]["revenue"]:,.2f} (42.36%), Tea: ${category_contribution.iloc[1]["revenue"]:,.2f} (21.62%). Top 2 = 63.98%. Smoothie: ${category_contribution.iloc[5]["revenue"]:,.2f} (6.93%)',
        'Recommendation': 'Diversifikasi revenue: (1) Perkuat Tea sebagai kategori #2 dengan varian baru (seasonal tea, tea-based smoothie). (2) Tingkatkan Smoothie melalui bundling dan promo. (3) Jaga Coffee dengan variasi limited-edition untuk menjaga novelty.',
        'Owner': 'Product Development Team',
        'Timeline': 'Strategic (3-12 bulan)',
        'Expected Impact': 'Reduksi kontribusi Coffee dari 42.36% ke 38-40% sambil meningkatkan Tea ke 23-25% dan Smoothie ke 8-10%',
        'Priority': 'Medium',
        'Risk': 'Investasi R&D untuk produk baru; risiko produk baru tidak diterima pasar',
        'KPI': 'Revenue Share per Category, Category Growth Rate, New Product Success Rate'
    },
    {
        'No': 'R8',
        'Problem': 'Average unit price sangat bervariasi antar produk ($1.95 - $14.65) tanpa strategi price ladder yang jelas',
        'Evidence': f'Range avg unit price: Chocolate Chip Cookie $1.95 s/d Tote Bag $14.65. Banyak produk clustered di range $2.50-$5.00 tanpa clear differentiation',
        'Recommendation': 'Implementasi price ladder yang jelas: (1) Value tier: $2-3 (cookie, small espresso). (2) Standard tier: $3.50-5 (cappuccino, latte). (3) Premium tier: $5-7 (matcha, specialty). (4) Ultra-premium: $10+ (merchandise). Jelasakan value proposition per tier.',
        'Owner': 'Pricing Manager',
        'Timeline': 'Mid Term (1-3 bulan)',
        'Expected Impact': 'Peningkatan clarity dalam product positioning; customer lebih mudah memilih berdasarkan budget, meningkatkan conversion rate',
        'Priority': 'Low',
        'Risk': 'Pelanggan existing mungkin confused dengan perubahan pricing; perlu gradual transition',
        'KPI': 'Revenue per Price Tier, Customer Upgrade Rate, Average Unit Price'
    }
])

print('Product Recommendations (4 rekomendasi):')
for _, row in product_recommendations.iterrows():
    print(f'\n[{row["No"]}] {row["Recommendation"][:100]}...')
    print(f'    Priority: {row["Priority"]} | Timeline: {row["Timeline"]}')

## 6.3 Customer Recommendations

In [ ]:
# ============================================================
# Customer Recommendations
# ============================================================

customer_recommendations = pd.DataFrame([
    {
        'No': 'R9',
        'Problem': f'Repeat customer rate sangat rendah ({repeat_rate:.2f}%); {one_time_customers:,} dari {n_customers:,} customers hanya transaksi sekali',
        'Evidence': f'Repeat rate: {repeat_rate:.2f}%. Customer segment High Value didominasi "One-time (1)" transaksi. Hanya {repeat_customers:,} customers ({repeat_rate:.2f}%) yang repeat',
        'Recommendation': 'Luncurkan loyalty program yang lebih agresif: (1) Welcome reward untuk first purchase. (2) Second purchase discount 15%. (3) Birthday reward. (4) Stamp card: "Buy 9 Get 1 Free". Target: mengubah 20% one-time customers menjadi repeat customers.',
        'Owner': 'Customer Relationship Manager',
        'Timeline': 'Quick Win (0-30 hari)',
        'Expected Impact': f'Peningkatan repeat rate dari {repeat_rate:.2f}% ke target 25% dalam 6 bulan; potensi revenue tambahan dari repeat customers yang signifikan',
        'Priority': 'High',
        'Risk': 'Biaya program loyalitas; pelanggan mungkin tidak merespons jika reward tidak menarik',
        'KPI': 'Repeat Customer Rate, Customer Lifetime Value, Loyalty Program Enrollment Rate'
    },
    {
        'No': 'R10',
        'Problem': f'Age group 25-34 mendominasi revenue ({age_group.iloc[0]["rev_pct"]:.2f}%) namun 65+ hanya {age_group.iloc[5]["rev_pct"]:.2f}%',
        'Evidence': f'25-34: ${age_group.iloc[0]["revenue"]:,.2f} ({age_group.iloc[0]["rev_pct"]:.2f}%), 35-44: ${age_group.iloc[1]["revenue"]:,.2f} ({age_group.iloc[1]["rev_pct"]:.2f}%). 65+: ${age_group.iloc[5]["revenue"]:,.2f} ({age_group.iloc[5]["rev_pct"]:.2f}%)',
        'Recommendation': 'Pertahankan dan deepen engagement dengan 25-34: exclusive early access, social media campaigns, app-based ordering. Untuk 65+: morning senior discount, comfortable seating area, larger print menu.',
        'Owner': 'Marketing Manager & Store Operations',
        'Timeline': 'Mid Term (1-3 bulan)',
        'Expected Impact': 'Peningkatan revenue dari 65+ sebesar 10-15% dan retensi 25-34 di level saat ini',
        'Priority': 'Medium',
        'Risk': 'Segmentasi terlalu granular bisa menghabiskan resources; perlu fokus pada high-impact segments',
        'KPI': 'Revenue per Age Group, Customer Engagement Rate per Segment, Acquisition Cost per Segment'
    },
    {
        'No': 'R11',
        'Problem': f'Seluruh customer adalah loyalty members (100%), namun repeat rate hanya {repeat_rate:.2f}% - program loyalitas belum efektif mendorong retensi',
        'Evidence': f'100% transaksi dari loyalty members (${loyalty_rev:,.2f}). Namun hanya {repeat_customers:,} dari {n_customers:,} customers melakukan repeat purchase. Program existing belum cukup untuk mendorong loyalitas aktual.',
        'Recommendation': 'Transform program loyalitas: (1) Tiered rewards berdasarkan frequency (Bronze/Silver/Gold). (2) Double points untuk repeat purchase dalam 7 hari. (3) Birthday reward. (4) Refer-a-friend reward. (5) Exclusive menu untuk tier Gold.',
        'Owner': 'CRM Team',
        'Timeline': 'Mid Term (1-3 bulan)',
        'Expected Impact': f'Peningkatan repeat rate dari {repeat_rate:.2f}% ke 15-25% dalam 6 bulan; peningkatan customer lifetime value',
        'Priority': 'Medium',
        'Risk': 'Cost of loyalty rewards; complexity program bisa membingungkan pelanggan',
        'KPI': 'Repeat Customer Rate, Customer Lifetime Value, Transaction Frequency per Customer'
    }
])

print('Customer Recommendations (3 rekomendasi):')
for _, row in customer_recommendations.iterrows():
    print(f'\n[{row["No"]}] {row["Recommendation"][:100]}...')
    print(f'    Priority: {row["Priority"]} | Timeline: {row["Timeline"]}')

## 6.4 Region and Store Recommendations

In [ ]:
# ============================================================
# Region and Store Recommendations
# ============================================================

region_recommendations = pd.DataFrame([
    {
        'No': 'R12',
        'Problem': f'Airport stores (4 dari 45, 8.9%) memiliki avg transaction tertinggi ($8.42) namun jumlah terbatas',
        'Evidence': f'Airport: avg txn $8.42, revenue ${store_type[store_type["store_type"]=="Airport"]["revenue"].values[0]:,.2f}. Standalone: avg txn $6.64, revenue ${store_type[store_type["store_type"]=="Standalone"]["revenue"].values[0]:,.2f}. Mall Kiosk: avg txn $6.83',
        'Recommendation': 'Ekspansi ke format Airport di kota baru. Target: tambah 2-3 airport stores dalam 12 bulan. Evaluasi lokasi airport dengan traffic tinggi. PertimbangkanAirport di negara yang belum terlayani (misal: CAN, UK airports).',
        'Owner': 'Business Development Director',
        'Timeline': 'Strategic (3-12 bulan)',
        'Expected Impact': 'Peningkatan total revenue 5-8% dari tambahan airport stores; peningkatan average transaction value per portfolio store',
        'Priority': 'Medium',
        'Risk': 'Biaya sewa airport yang tinggi; regulasi dan izin operasional; kompetisi ketat',
        'KPI': 'Revenue per Store Type, Airport Store ROI, New Store Time-to-Profitability'
    },
    {
        'No': 'R13',
        'Problem': f'Store berkinerja rendah (bottom 5: Manchester #21 $2,077, Manchester #22 $2,415, Manchester #24 $2,539, Manchester #23 $2,571, London #16 $2,509)',
        'Evidence': f'Bottom 5 semuanya di UK (Manchester/London). Manchester #21 avg txn $5.28 (terendah portfolio). Revenue gap top-bottom: ${top5_stores.iloc[0]["revenue"]-bot5_stores.iloc[0]["revenue"]:,.2f}',
        'Recommendation': 'Audit menyeluruh 5 store terbawah: (1) Evaluasi lokasi dan foot traffic. (2) Training staff. (3) Implementasi best practices dari top performers. (4) Jika tidak membaik dalam 6 bulan, pertimbangkan restrukturisasi atau penutupan.',
        'Owner': 'Regional Manager (UK)',
        'Timeline': 'Quick Win (0-30 hari)',
        'Expected Impact': 'Peningkatan revenue UK stores 10-20% atau pengambilan keputusan strategis untuk realokasi resources',
        'Priority': 'High',
        'Risk': 'Penutupan store berdampak pada brand presence di UK; perlu data cost operasional untuk keputusan final',
        'KPI': 'Revenue per Store, Avg Transaction Value per Store, Store Performance Score'
    },
    {
        'No': 'R14',
        'Problem': f'Revenue gap antar negara sangat lebar: USA $47,694.86 vs UK $25,772.96 (gap ${country_rev_range:,.2f})',
        'Evidence': f'USA: 34.81%, AUS: 24.28%, CAN: 22.10%, UK: 18.81%. UK underperform secara konsisten across all cities',
        'Recommendation': 'Strategi turnaround untuk UK: (1) Benchmark best practices dari USA/AUS. (2) Lokalisasi menu sesuai preferensi lokal. (3) Evaluasi store placement. (4) Marketing campaign khusus UK market. Target: naikkan UK share ke 20%+ dalam 12 bulan.',
        'Owner': 'VP International Operations',
        'Timeline': 'Strategic (3-12 bulan)',
        'Expected Impact': 'Peningkatan UK revenue 10-15% dan reduksi gap antar negara',
        'Priority': 'Medium',
        'Risk': 'Perbedaan cultural taste; biaya turnaround tinggi; kompetisi lokal ketat',
        'KPI': 'Revenue Share per Country, Country Growth Rate, Store Performance Index per Country'
    }
])

print('Region & Store Recommendations (3 rekomendasi):')
for _, row in region_recommendations.iterrows():
    print(f'\n[{row["No"]}] {row["Recommendation"][:100]}...')
    print(f'    Priority: {row["Priority"]} | Timeline: {row["Timeline"]}')

## 6.5 Time Recommendations

In [ ]:
# ============================================================
# Time Recommendations
# ============================================================

time_recommendations = pd.DataFrame([
    {
        'No': 'R15',
        'Problem': f'Peak hour (7:00-10:00) sangat padat ({hourly_summary[hourly_summary["hour"].isin([7,8,9])]["transactions"].sum():,} transaksi); risk understaffing dan service degradation',
        'Evidence': f'Jam 7: 1,834 txn, Jam 8: 2,243 txn (PEAK), Jam 9: 1,879 txn. Total pagi: ~5,956 transaksi. Night hours (0-5): rata-rata ~224 transaksi/jam',
        'Recommendation': 'Dynamic staffing: tambah 30-50% staff di jam 7-10. Implementasi shift khusus morning rush. Siapkan prep station khusus untuk order populer (cappuccino, latte). Target: waktu tunggu customer < 3 menit di peak hour.',
        'Owner': 'Operations Manager',
        'Timeline': 'Quick Win (0-30 hari)',
        'Expected Impact': 'Peningkatan throughput 15-20% di peak hour; pengurangan customer complaints; peningkatan customer satisfaction',
        'Priority': 'High',
        'Risk': 'Overstaffing jika prediksi salah; biaya overtime; staff burnout',
        'KPI': 'Peak Hour Transaction Volume, Average Wait Time, Staff-to-Transaction Ratio, Customer Satisfaction Score'
    },
    {
        'No': 'R16',
        'Problem': f'Off-peak hours (0:00-6:00) sangat sepi (rata-rata ~224 txn/jam), namun tetap beroperasi',
        'Evidence': f'Jam 0: 369 txn, Jam 1: 182 txn, Jam 2: 165 txn, Jam 3: 205 txn, Jam 4: 203 txn, Jam 5: 386 txn. Total: ~1,510 transaksi dari 6 jam',
        'Recommendation': 'Optimasi off-peak: (1) Promosi khusus off-peak ("Night Owl Discount" jam 0-5). (2) Evaluasi cost-benefit apakah semua store perlu buka 24 jam. (3) Delivery/online order focus di jam sepi.',
        'Owner': 'Operations Manager & Marketing',
        'Timeline': 'Mid Term (1-3 bulan)',
        'Expected Impact': 'Peningkatan off-peak transaction 10-20% atau pengurangan biaya operasional jika mengurangi jam operasional',
        'Priority': 'Low',
        'Risk': 'Off-peak customers mungkin sensitif harga; pengurangan jam operasional berdampak pada brand presence',
        'KPI': 'Off-Peak Transaction Volume, Revenue per Operating Hour, Cost per Transaction'
    },
    {
        'No': 'R17',
        'Problem': f'Weekday revenue per hari ($17,349.26/5 hari = $3,469.85/hari) jauh lebih rendah dari weekend ($25,131.50/2 hari = $12,565.75/hari)',
        'Evidence': f'Weekday total: $86,746.28 (5 hari). Weekend total: $50,262.99 (2 hari). Avg daily weekday: ${avg_weekday_rev_per_day:,.2f}. Avg daily weekend: ${avg_weekend_rev_per_day:,.2f}. Weekend lift: +{weekend_lift_pct:.1f}%',
        'Recommendation': 'Weekday promotion strategy: (1) Monday "Fresh Start" discount. (2) Wednesday "Mid-Week Pick-Me-Up" bundle. (3) Corporate catering/packages for office workers. (4) Loyalty double points on weekdays.',
        'Owner': 'Marketing Manager',
        'Timeline': 'Quick Win (0-30 hari)',
        'Expected Impact': 'Peningkatan weekday revenue 8-12% dari baseline; pengurangan gap weekday-weekend',
        'Priority': 'High',
        'Risk': 'Over-discounting weekdays bisa mengurangi margin; perlu A/B testing untuk menentukan format promosi optimal',
        'KPI': 'Weekday vs Weekend Revenue Ratio, Monday Transaction Volume, Mid-Week Promotion ROI'
    }
])

print('Time Recommendations (3 rekomendasi):')
for _, row in time_recommendations.iterrows():
    print(f'\n[{row["No"]}] {row["Recommendation"][:100]}...')
    print(f'    Priority: {row["Priority"]} | Timeline: {row["Timeline"]}')

## 6.6 Discount Recommendations

In [ ]:
# ============================================================
# Discount Recommendations
# ============================================================

discount_recommendations = pd.DataFrame([
    {
        'No': 'R18',
        'Problem': f'Diskon sudah digunakan pada {pct_discount_txn:.2f}% transaksi namun tanpa framework terstruktur',
        'Evidence': f'Transaksi diskon: {len(disc_applied):,} dari {total_transactions:,} ({pct_discount_txn:.2f}%). Avg transaction diskon: ${disc_avg_txn:.2f}. Avg tanpa diskon: ${no_disc_avg_txn:.2f}',
        'Recommendation': 'Implementasi discount framework: (1) Discount hanya untuk produk tertentu (high-stock, seasonal ending). (2) Minimum purchase untuk diskon (misal: belanja >$10 untuk diskon 10%). (3) Time-based discount (off-peak only). (4) Loyalty-exclusive discounts.',
        'Owner': 'Revenue Manager',
        'Timeline': 'Quick Win (0-30 hari)',
        'Expected Impact': 'Peningkatan efektivitas diskon: volume naik 5-10% tanpa penurunan revenue per transaksi yang signifikan',
        'Priority': 'High',
        'Risk': 'Pelanggan terbiasa diskon dan menunggu promo; perlu gradual implementation',
        'KPI': 'Discount Transaction Rate, Revenue per Discounted Transaction, Volume Lift per Discount Campaign'
    },
    {
        'No': 'R19',
        'Problem': 'Tidak ada segmentasi diskon berdasarkan customer segment; diskon diberikan secara merata',
        'Evidence': 'Customer segment High Value memiliki avg transaction $11.95 vs Low Value $2.91. Diskon pada High Value mungkin tidak diperlukan, diskon pada Low Value mungkin lebih efektif untuk meningkatkan frekuensi',
        'Recommendation': 'Segmentasi diskon: (1) High Value: loyalty rewards, bukan diskon langsung. (2) Medium Value: targeted discount untuk upgrade ke High Value. (3) Low Value: aggressive discount untuk meningkatkan frequency dan basket size.',
        'Owner': 'CRM & Revenue Manager',
        'Timeline': 'Mid Term (1-3 bulan)',
        'Expected Impact': 'Peningkatan customer value dari Medium ke High sebesar 10-15%; penghematan biaya diskon 5-10%',
        'Priority': 'Medium',
        'Risk': 'High Value customers merasa "dihadiahi" lebih sedikit; data customer segment perlu diupdate secara berkala',
        'KPI': 'Discount ROI per Customer Segment, Customer Upgrade Rate, Discount Cost as % of Revenue'
    }
])

print('Discount Recommendations (2 rekomendasi):')
for _, row in discount_recommendations.iterrows():
    print(f'\n[{row["No"]}] {row["Recommendation"][:100]}...')
    print(f'    Priority: {row["Priority"]} | Timeline: {row["Timeline"]}')

---
# 7. Quick Wins vs Strategic Actions

## 7.1 Quick Wins (0-30 hari)

In [ ]:
# ============================================================
# Quick Wins Summary
# ============================================================

quick_wins = pd.DataFrame([
    {'Recommendation': 'R1: Bundle & Upselling di POS', 'Category': 'Sales', 'Owner': 'Head of Sales', 'Timeline': '0-15 hari', 'KPI': 'ATV, Items per Transaction', 'Priority': 'High'},
    {'Recommendation': 'R15: Dynamic Staffing Peak Hour (7-10)', 'Category': 'Operations', 'Owner': 'Operations Manager', 'Timeline': '0-15 hari', 'KPI': 'Wait Time, Throughput', 'Priority': 'High'},
    {'Recommendation': 'R17: Weekday Promotion Campaign', 'Category': 'Marketing', 'Owner': 'Marketing Manager', 'Timeline': '0-30 hari', 'KPI': 'Weekday Revenue, Transaction Volume', 'Priority': 'High'},
    {'Recommendation': 'R9: Enhanced Loyalty Program Launch', 'Category': 'Customer', 'Owner': 'CRM Manager', 'Timeline': '0-30 hari', 'KPI': 'Repeat Rate, Loyalty Enrollment', 'Priority': 'High'},
    {'Recommendation': 'R18: Discount Framework Implementation', 'Category': 'Pricing', 'Owner': 'Revenue Manager', 'Timeline': '0-30 hari', 'KPI': 'Discount Rate, Revenue per Discounted Txn', 'Priority': 'High'},
    {'Recommendation': 'R5: Merchandise Add-on at Checkout', 'Category': 'Product', 'Owner': 'Product Manager', 'Timeline': '0-15 hari', 'KPI': 'Merchandise Quantity, Add-on Rate', 'Priority': 'High'},
    {'Recommendation': 'R13: UK Store Performance Audit', 'Category': 'Region', 'Owner': 'Regional Manager UK', 'Timeline': '0-30 hari', 'KPI': 'Revenue per Store, ATV per Store', 'Priority': 'High'},
])

print('=' * 100)
print(' QUICK WINS (0-30 hari)')
print('=' * 100)
print(quick_wins.to_string(index=False))
print(f'\nTotal Quick Wins: {len(quick_wins)} rekomendasi')

## 7.2 Strategic Actions (1-12 bulan)

In [ ]:
# ============================================================
# Strategic Actions Summary
# ============================================================

strategic_actions = pd.DataFrame([
    {'Recommendation': 'R2: Seasonal Campaign Bulan Rendah', 'Category': 'Marketing', 'Owner': 'Marketing Manager', 'Timeline': '1-3 bulan', 'KPI': 'Monthly Revenue Gap', 'Priority': 'Medium'},
    {'Recommendation': 'R7: Category Diversification (Tea & Smoothie)', 'Category': 'Product', 'Owner': 'Product Development', 'Timeline': '3-12 bulan', 'KPI': 'Category Revenue Share', 'Priority': 'Medium'},
    {'Recommendation': 'R12: Airport Store Expansion', 'Category': 'Region', 'Owner': 'Business Development', 'Timeline': '3-12 bulan', 'KPI': 'Revenue per Store Type, New Store ROI', 'Priority': 'Medium'},
    {'Recommendation': 'R14: UK Market Turnaround', 'Category': 'Region', 'Owner': 'VP International', 'Timeline': '3-12 bulan', 'KPI': 'Country Revenue Share, UK Growth Rate', 'Priority': 'Medium'},
    {'Recommendation': 'R10: Age Group Targeted Marketing', 'Category': 'Customer', 'Owner': 'Marketing Manager', 'Timeline': '1-3 bulan', 'KPI': 'Revenue per Age Group', 'Priority': 'Medium'},
    {'Recommendation': 'R11: Loyalty Program Enhancement', 'Category': 'Customer', 'Owner': 'CRM Team', 'Timeline': '1-3 bulan', 'KPI': 'Loyalty Enrollment, Transaction Frequency', 'Priority': 'Medium'},
])

print('=' * 110)
print(' STRATEGIC ACTIONS (1-12 bulan)')
print('=' * 110)
print(strategic_actions.to_string(index=False))
print(f'\nTotal Strategic Actions: {len(strategic_actions)} rekomendasi')

---
# 8. Top 3 Priority Recommendations

## Priority 1: Program Loyalitas & Repeat Customer Recovery

**Rekomendasi R9**

| Komponen | Detail |
|---|---|
| **Alasan Dipilih** | Repeat customer rate 3.04% adalah masalah paling kritis. Dari 19,250 customers, hanya 586 yang melakukan repeat purchase. Ini berarti 96.96% pelanggan hilang setelah transaksi pertama.|
| **Bukti Angka** | 19,250 unique customers, 586 repeat customers (3.04%), 18,664 one-time customers (96.96%). Customer segment High Value masih didominasi "One-time (1)" transaksi.|
| **Dampak Bisnis** | Jika 20% dari 18,664 one-time customers bisa diubah menjadi repeat (3,733 customers), dan masing-masing menghabiskan rata-rata $30/tahun (setara ~4 transaksi), potensi revenue tambahan: 3,733 x $30 = $111,990.|
| **Urgensi** | Sangat tinggi. Setiap hari tanpa program loyalitas berarti kehilangan pelanggan yang tidak kembali.|
| **Risiko jika Tidak Dilakukan** | Persaingan semakin ketat; competitors akan menarik customers kita. Customer acquisition cost akan terus naik tanpa retensi.|
| **KPI yang Dipantau** | Repeat Customer Rate (target: 25% dalam 6 bulan), Loyalty Program Enrollment Rate, Customer Lifetime Value, Monthly Repeat Transaction Count|

---

## Priority 2: Peak Hour Operations Optimization

**Rekomendasi R15**

| Komponen | Detail |
|---|---|
| **Alasan Dipilih** | Jam 7-10 pagi menyumbang ~30% dari total transaksi. Ketidakseimbangan staff di jam ini langsung mempengaruhi customer experience dan revenue.|
| **Bukti Angka** | Jam 8: 2,243 transaksi ($18,213.56 revenue). Jam 7-9 total: ~5,956 transaksi. Night hours: ~224 transaksi/jam. Rasio peak vs off-peak: 10:1.|
| **Dampak Bisnis** | Peningkatan throughput 15-20% di peak hour = ~900-1,200 transaksi tambahan per hari di jam sibuk. Dengan ATV $6.85, potensi revenue tambahan: 1,000 x $6.85 = $6,850/hari atau ~$2.5M/tahun.|
| **Urgensi** | Tinggi. Peak hour inefficiency langsung berdampak pada revenue harian.|
| **Risiko jika Tidak Dilakukan** | Customer complaints meningkat; antrian panjang; customers beralih ke kompetitor. Loss of revenue dari customers yang tidak jadi beli karena waiting time lama.|
| **KPI yang Dipantau** | Average Wait Time (target: <3 menit), Peak Hour Throughput, Staff-to-Transaction Ratio, Customer Satisfaction Score|

---

## Priority 3: ATV Enhancement through Bundling & Upselling

**Rekomendasi R1**

| Komponen | Detail |
|---|---|
| **Alasan Dipilih** | Dari 20,000 transaksi, ATV hanya $6.85. Banyak transaksi single-item. Peningkatan basket size adalah quick win dengan dampak langsung pada revenue.|
| **Bukti Angka** | ATV: $6.85, median lebih rendah dari mean (right-skewed). Rata-rata quantity: 1.71 unit/transaksi. Banyak produk $2-5 yang bisa menjadi add-on.|
| **Dampak Bisnis** | Peningkatan ATV 5-10% = $0.34-$0.69 per transaksi. Dari 20,000 transaksi: $6,800-$13,800 potensi revenue tambahan per tahun.|
| **Urgensi** | Medium-High. Quick win yang bisa dimulai segera tanpa investasi besar.|
| **Risiko jika Tidak Dilakukan** | Revenue stagnan meskipun volume transaksi naik. Competitors lebih agresif dalam upselling.|
| **KPI yang Dipantau** | Average Transaction Value (target: $7.20-$7.50), Items per Transaction, Bundle Conversion Rate, Upsell Success Rate|

---

---
# 9. Expected Business Impact

## 9.1 Estimasi Dampak

**Penting:** Dataset ini **tidak menyediakan data cost, profit, atau margin**. Estimasi dampak berikut menggunakan **revenue, transaction count, quantity, dan ATV** sebagai dasar. Dampak bersifat **skenario**, bukan hasil pasti.

### Actual Data
- Total Revenue Baseline: $137,009.27
- Total Transactions: 20,000
- Average Transaction Value: $6.85
- Repeat Customer Rate: 3.04%

In [ ]:
# ============================================================
# Estimasi Dampak Bisnis
# ============================================================

print('=' * 80)
print(' ESTIMASI DAMPAK BISNIS (SCENARIO-BASED)')
print('=' * 80)

baseline_revenue = total_revenue
baseline_txn = total_transactions
baseline_atv = avg_transaction_value
baseline_repeat = repeat_rate

print(f'\n--- BASELINE ---')
print(f'Total Revenue:      ${baseline_revenue:,.2f}')
print(f'Total Transactions:  {baseline_txn:,}')
print(f'ATV:                ${baseline_atv:.2f}')
print(f'Repeat Rate:        {baseline_repeat:.2f}%')

print(f'\n--- SCENARIO 1: ATV Enhancement (Rekomendasi R1) ---')
print(f'Assumption: Peningkatan ATV sebesar 5-10% dari baseline ${baseline_atv:.2f}')
scenario1_low = baseline_revenue * 0.05
scenario1_high = baseline_revenue * 0.10
print(f'Estimasi revenue tambahan: ${scenario1_low:,.2f} - ${scenario1_high:,.2f}')
print(f'New revenue range: ${baseline_revenue + scenario1_low:,.2f} - ${baseline_revenue + scenario1_high:,.2f}')

print(f'\n--- SCENARIO 2: Repeat Customer Improvement (Rekomendasi R9) ---')
print(f'Assumption: 20% dari 1-time customers ({one_time_customers:,}) menjadi repeat, avg 4 transaksi/tahun')
new_repeat = int(one_time_customers * 0.20)
additional_txn = new_repeat * 4
additional_rev = additional_txn * baseline_atv
print(f'New repeat customers: {new_repeat:,}')
print(f'Additional transactions: {additional_txn:,}')
print(f'Estimated additional revenue: ${additional_rev:,.2f}')

print(f'\n--- SCENARIO 3: Weekday Revenue Lift (Rekomendasi R17) ---')
print(f'Assumption: Peningkatan weekday revenue 8-12% dari baseline ${weekday_data["revenue"].values[0]:,.2f}')
weekday_lift_low = weekday_data['revenue'].values[0] * 0.08
weekday_lift_high = weekday_data['revenue'].values[0] * 0.12
print(f'Estimated weekday revenue lift: ${weekday_lift_low:,.2f} - ${weekday_lift_high:,.2f}')

print(f'\n--- SCENARIO 4: Peak Hour Throughput Improvement (Rekomendasi R15) ---')
print(f'Assumption: Peningkatan throughput 15-20% di jam 7-10')
peak_txn = hourly_summary[hourly_summary['hour'].isin([7,8,9])]['transactions'].sum()
peak_throughput_low = int(peak_txn * 0.15)
peak_throughput_high = int(peak_txn * 0.20)
peak_rev_low = peak_throughput_low * baseline_atv
peak_rev_high = peak_throughput_high * baseline_atv
print(f'Peak hour current transactions: {peak_txn:,}')
print(f'Additional transactions: {peak_throughput_low:,} - {peak_throughput_high:,}')
print(f'Estimated revenue lift: ${peak_rev_low:,.2f} - ${peak_rev_high:,.2f}')

print(f'\n--- COMBINED IMPACT (SCENARIO) ---')
total_low = scenario1_low + additional_rev + weekday_lift_low + peak_rev_low
total_high = scenario1_high + additional_rev + weekday_lift_high + peak_rev_high
print(f'Conservative estimate: ${total_low:,.2f}')
print(f'Optimistic estimate:   ${total_high:,.2f}')
print(f'New revenue range:     ${baseline_revenue + total_low:,.2f} - ${baseline_revenue + total_high:,.2f}')
print(f'\nCATATAN: Estimasi ini bersifat skenario. Dampak aktual bergantung pada kualitas implementasi, kondisi pasar, dan faktor eksternal lainnya.')

---
# 10. Risk and Mitigation

In [ ]:
# ============================================================
# Risk and Mitigation Table
# ============================================================

risk_table = pd.DataFrame([
    {
        'Recommendation': 'R1: Bundle & Upselling',
        'Potential Risk': 'Cannibalization produk high-margin; pelanggan merasa "dijual" terus-menerus',
        'Mitigation': 'A/B testing berbagai format bundling; limit promosi ke maximum 1x per customer per hari; monitor customer feedback'
    },
    {
        'Recommendation': 'R5: Merchandise Add-on',
        'Potential Risk': 'Overstock merchandise jika permintaan tidak naik; inventori menumpuk',
        'Mitigation': 'Monitor stok merchandise secara mingguan; reorder point berdasarkan actual sales velocity; gunakan just-in-time ordering'
    },
    {
        'Recommendation': 'R9: Loyalty Program',
        'Potential Risk': 'Biaya loyalty rewards tinggi; program tidak menarik bagi pelanggan',
        'Mitigation': 'Budget cap untuk loyalty rewards; feedback loop dengan customers; iterate program berdasarkan data engagement'
    },
    {
        'Recommendation': 'R13: UK Store Audit',
        'Potential Risk': 'Penutupan store berdampak pada brand presence di UK; pelanggan loyal kecewa',
        'Mitigation': 'Evaluasi berbasis data 6 bulan; komunikasi terbuka dengan pelanggan; pertimbangkan hybrid model (smaller format)'
    },
    {
        'Recommendation': 'R15: Dynamic Staffing',
        'Potential Risk': 'Overstaffing jika prediksi salah; biaya overtime meningkat',
        'Mitigation': 'Gunakan data historical untuk prediksi akurat; implementasi bertahap; cross-training staff untuk fleksibilitas'
    },
    {
        'Recommendation': 'R17: Weekday Promotion',
        'Potential Risk': 'Over-discounting weekdays; customers menunggu diskon dan tidak beli di harga normal',
        'Mitigation': 'Limited-time offers (1-2 minggu); rotasi jenis promosi; batasi diskon maksimum 15%; A/B testing'
    },
    {
        'Recommendation': 'R12: Airport Expansion',
        'Potential Risk': 'Biaya sewa airport sangat tinggi; break-even lebih lama dari prediksi',
        'Mitigation': 'Due diligence ketat sebelum signing lease; target airport dengan >80% occupancy rate; negosiasi revenue-share model'
    },
    {
        'Recommendation': 'R18: Discount Framework',
        'Potential Risk': 'Pelanggan terbiasa diskon menolak harga normal; volume transaksi turun',
        'Mitigation': 'Gradual reduction diskon; komunikasi value proposition; tawarkan alternatif non-diskon (loyalty points, exclusive access)'
    },
    {
        'Recommendation': 'R7: Category Diversification',
        'Potential Risk': 'Produk baru tidak diterima pasar; investasi R&D sia-sia',
        'Mitigation': 'Pilot testing di 3-5 store sebelum rollout; customer taste testing; limit initial investment'
    },
    {
        'Recommendation': 'General: Data Limitation',
        'Potential Risk': 'Rekomendasi berbasis data yang tidak memiliki profit/cost; ada risk rekomendasi tidak optimal dari sisi profitability',
        'Mitigation': 'Lengkapi dataset dengan cost data; lakukan profit analysis sebelum eksekusi besar; prioritize rekomendasi berdasarkan revenue impact yang sudah terverifikasi'
    }
])

print('=' * 120)
print(' RISK AND MITIGATION TABLE')
print('=' * 120)
for _, row in risk_table.iterrows():
    print(f'\n--- {row["Recommendation"]} ---')
    print(f'  Risk:       {row["Potential Risk"]}')
    print(f'  Mitigation: {row["Mitigation"]}')

---
# 11. KPI Monitoring Plan

In [ ]:
# ============================================================
# KPI Monitoring Plan
# ============================================================

kpi_plan = pd.DataFrame([
    {
        'KPI': 'Total Revenue',
        'Definition': 'Total seluruh revenue dari semua transaksi dalam periode',
        'Frequency': 'Mingguan & Bulanan',
        'Owner': 'Finance & Sales Director',
        'Target/Baseline': f'Baseline: ${total_revenue:,.2f}/periode data. Target: naik 5-10% QoQ'
    },
    {
        'KPI': 'Transaction Count',
        'Definition': 'Jumlah total transaksi unik dalam periode',
        'Frequency': 'Mingguan & Bulanan',
        'Owner': 'Operations Manager',
        'Target/Baseline': f'Baseline: {total_transactions:,} transaksi. Target: naik 3-5% QoQ'
    },
    {
        'KPI': 'Quantity Sold',
        'Definition': 'Total unit produk yang terjual dalam periode',
        'Frequency': 'Mingguan',
        'Owner': 'Inventory Manager',
        'Target/Baseline': f'Baseline: {total_quantity:,} unit. Target: naik seiring pertumbuhan transaksi'
    },
    {
        'KPI': 'Average Transaction Value',
        'Definition': 'Total revenue dibagi jumlah transaksi',
        'Frequency': 'Mingguan & Bulanan',
        'Owner': 'Revenue Manager',
        'Target/Baseline': f'Baseline: ${avg_transaction_value:.2f}. Target: $7.20-$7.50'
    },
    {
        'KPI': 'Repeat Customer Rate',
        'Definition': 'Persentase customers yang melakukan >1 transaksi',
        'Frequency': 'Bulanan',
        'Owner': 'CRM Manager',
        'Target/Baseline': f'Baseline: {repeat_rate:.2f}%. Target: 25% dalam 6 bulan'
    },
    {
        'KPI': 'Revenue per Store',
        'Definition': 'Revenue per store dalam periode',
        'Frequency': 'Bulanan',
        'Owner': 'Regional Manager',
        'Target/Baseline': f'Baseline: avg ${total_revenue/n_stores:,.2f}/store. Target: penurunan gap top-bottom 20%'
    },
    {
        'KPI': 'Revenue per Product Category',
        'Definition': 'Revenue per kategori produk dalam periode',
        'Frequency': 'Bulanan',
        'Owner': 'Product Manager',
        'Target/Baseline': 'Baseline: Coffee 42.36%, Tea 21.62%. Target: Coffee <40%, Tea >23%, Smoothie >8%'
    },
    {
        'KPI': 'Peak Hour Transactions',
        'Definition': 'Jumlah transaksi di jam 7-10 pagi',
        'Frequency': 'Mingguan',
        'Owner': 'Operations Manager',
        'Target/Baseline': f'Baseline: ~{hourly_summary[hourly_summary["hour"].isin([7,8,9])]["transactions"].sum():,} transaksi. Target: naik 15-20%'
    },
    {
        'KPI': 'Discount Transaction Rate',
        'Definition': 'Persentase transaksi yang menggunakan diskon',
        'Frequency': 'Bulanan',
        'Owner': 'Revenue Manager',
        'Target/Baseline': f'Baseline: {pct_discount_txn:.2f}%. Target: stabil atau turun sambil menjaga volume'
    },
    {
        'KPI': 'Weekday vs Weekend Revenue Ratio',
        'Definition': 'Rasio revenue per hari weekday vs weekend',
        'Frequency': 'Mingguan',
        'Owner': 'Marketing Manager',
        'Target/Baseline': f'Baseline: weekday ${avg_weekday_rev_per_day:,.2f}/day vs weekend ${avg_weekend_rev_per_day:,.2f}/day. Target: reduksi gap 8-12%'
    },
    {
        'KPI': 'Loyalty Program Effectiveness',
        'Definition': 'Repeat rate dari loyalty members (program sudah mencakup 100% customers)',
        'Frequency': 'Bulanan',
        'Owner': 'CRM Manager',
        'Target/Baseline': f'Baseline: {repeat_rate:.2f}% repeat rate. Target: 15-25% dalam 6 bulan'
    }
])

print('=' * 120)
print(' KPI MONITORING PLAN')
print('=' * 120)
for _, row in kpi_plan.iterrows():
    print(f'\n--- {row["KPI"]} ---')
    print(f'  Definition:  {row["Definition"]}')
    print(f'  Frequency:   {row["Frequency"]}')
    print(f'  Owner:       {row["Owner"]}')
    print(f'  Target/Base: {row["Target/Baseline"]}')

---
# 12. Bonus Challenge

## Option C: Dua Cabang Harus Ditutup

**Pertanyaan:** Jika CEO menginstruksikan penutupan 2 cabang, cabang mana yang harus ditutup dan apa dasarnya?

### Analisis

In [ ]:
# ============================================================
# Bonus Challenge: Store Closure Analysis
# ============================================================

print('=' * 100)
print(' BONUS CHALLENGE: STORE CLOSURE ANALYSIS')
print('=' * 100)

# Sort stores by revenue ascending
stores_sorted = store_summary.sort_values('revenue', ascending=True).reset_index(drop=True)

print(f'\n--- Bottom 10 Stores (Candidate for Closure) ---')
print(f'{"Rank":<5} {"Store":<10} {"City":<15} {"Type":<15} {"Revenue":>12} {"Txn":>8} {"Qty":>8} {"Avg Txn":>10}')
print('-' * 85)
for i, (_, row) in enumerate(stores_sorted.head(10).iterrows(), 1):
    print(f'{i:<5} #{int(row["store_id"]):<9} {row["city"]:<15} {row["store_type"]:<15} ${row["revenue"]:>10,.2f} {int(row["transactions"]):>8} {int(row["quantity"]):>8} ${row["avg_transaction"]:>8.2f}')

print(f'\n--- Top 5 Stores (Reference) ---')
for i, (_, row) in enumerate(stores_sorted.tail(5).iterrows(), 1):
    print(f'  {i}. #{int(row["store_id"])} {row["city"]} ({row["store_type"]}): ${row["revenue"]:,.2f}, avg txn ${row["avg_transaction"]:.2f}')

# Analysis of bottom 2
bottom2 = stores_sorted.head(2)
print(f'\n--- RECOMMENDATION: 2 Stores to Close ---')
for _, row in bottom2.iterrows():
    print(f'\n  Store #{int(row["store_id"])} - {row["city"]} ({row["store_type"]})')
    print(f'    Revenue: ${row["revenue"]:,.2f} (terendah portfolio)')
    print(f'    Transactions: {int(row["transactions"]):,}')
    print(f'    Quantity: {int(row["quantity"]):,}')
    print(f'    Avg Transaction: ${row["avg_transaction"]:.2f} (terendah portfolio)')
    
    # Compare with portfolio average
    avg_portfolio = store_summary['revenue'].mean()
    avg_txn_portfolio = store_summary['avg_transaction'].mean()
    rev_deficit = avg_portfolio - row['revenue']
    txn_deficit = avg_txn_portfolio - row['avg_transaction']
    print(f'    Deficit vs Portfolio Average:')
    print(f'      Revenue: -${rev_deficit:,.2f} (below avg by {rev_deficit/avg_portfolio*100:.1f}%)')
    print(f'      Avg Transaction: -${txn_deficit:.2f} (below avg by {txn_deficit/avg_txn_portfolio*100:.1f}%)')

# What if these stores are closed
closed_rev = bottom2['revenue'].sum()
closed_txn = bottom2['transactions'].sum()
closed_qty = bottom2['quantity'].sum()
remaining_stores = len(store_summary) - 2

print(f'\n--- Impact of Closing These 2 Stores ---')
print(f'  Revenue lost: ${closed_rev:,.2f} ({closed_rev/total_revenue*100:.2f}% of total)')
print(f'  Transactions lost: {int(closed_txn):,} ({closed_txn/total_transactions*100:.2f}% of total)')
print(f'  Quantity lost: {int(closed_qty):,} ({closed_qty/total_quantity*100:.2f}% of total)')
print(f'  Remaining stores: {remaining_stores}')
print(f'  Average revenue per remaining store: ${(total_revenue - closed_rev)/remaining_stores:,.2f}')

print(f'\n--- Justification ---')
print(f'  1. Both stores have the lowest revenue in the portfolio')
print(f'  2. Both have the lowest average transaction value (< $5.50)')
print(f'  3. Combined revenue loss is only {closed_rev/total_revenue*100:.2f}% of total')
print(f'  4. Resources can be reallocated to higher-performing stores or new airport locations')
print(f'  5. Both are in Manchester (same city), reducing geographic redundancy')

print(f'\n--- CATATAN PENTING ---')
print(f'  Keputusan penutupan final MEMERLUKAN data biaya operasional dan profit per store.')
print(f'  Dataset ini hanya menyediakan revenue; tidak ada data cost, sewa, gaji, atau profit.')
print(f'  Rekomendasi ini harus divalidasi dengan data keuangan lengkap sebelum eksekusi.')

---
# 13. Final Recommendation Table

Semua rekomendasi diurutkan berdasarkan prioritas.

In [ ]:
# ============================================================
# Final Recommendation Table
# ============================================================

all_recommendations = pd.concat([
    sales_recommendations[['No', 'Recommendation', 'Evidence', 'Priority', 'Owner', 'Timeline', 'KPI']],
    product_recommendations[['No', 'Recommendation', 'Evidence', 'Priority', 'Owner', 'Timeline', 'KPI']],
    customer_recommendations[['No', 'Recommendation', 'Evidence', 'Priority', 'Owner', 'Timeline', 'KPI']],
    region_recommendations[['No', 'Recommendation', 'Evidence', 'Priority', 'Owner', 'Timeline', 'KPI']],
    time_recommendations[['No', 'Recommendation', 'Evidence', 'Priority', 'Owner', 'Timeline', 'KPI']],
    discount_recommendations[['No', 'Recommendation', 'Evidence', 'Priority', 'Owner', 'Timeline', 'KPI']],
], ignore_index=True)

# Sort by priority
priority_order = {'High': 0, 'Medium': 1, 'Low': 2}
all_recommendations['priority_rank'] = all_recommendations['Priority'].map(priority_order)
all_recommendations = all_recommendations.sort_values('priority_rank').drop('priority_rank', axis=1).reset_index(drop=True)
all_recommendations.index = all_recommendations.index + 1
all_recommendations.index.name = 'Rank'

print('=' * 130)
print(' FINAL RECOMMENDATION TABLE (Sorted by Priority)')
print('=' * 130)
for rank, (_, row) in enumerate(all_recommendations.iterrows(), 1):
    print(f'\n  Rank {rank} [{row["No"]}] ({row["Priority"]})')
    print(f'  Recommendation: {row["Recommendation"][:90]}')
    print(f'  Owner: {row["Owner"]} | Timeline: {row["Timeline"]}')
    print(f'  KPI: {row["KPI"][:80]}')
    print(f'  -' * 50)

---
# 14. Conclusion

## Masalah Bisnis Utama

Berdasarkan analisis menyeluruh dari 20,000 transaksi di 45 store dengan 43 produk, masalah bisnis utama yang teridentifikasi adalah:

1. **Repeat customer rate yang sangat rendah (3.04%)** - 96.96% pelanggan tidak kembali setelah transaksi pertama
2. **Average Transaction Value yang masih rendah ($6.85)** - banyak transaksi single-item dengan potensi upselling besar
3. **Performa temporal yang tidak seimbang** - weekend revenue per hari 3.6x lipat lebih tinggi dari weekday; peak hour (7-10am) sangat padat sementara off-peak sangat sepi

## Penyebab dan Pola Utama

- **Pola musiman** terlihat dari gap bulanan sebesar $1,394 (Januari vs Maret)
- **Konsentrasi revenue** pada 2 kategori (Coffee 42.36% + Tea 21.62% = 63.98%)

## Tiga Tindakan Prioritas

1. **Program Loyalitas Agresif** untuk mengubah one-time customers menjadi repeat customers
2. **Optimalisasi Peak Hour Operations** untuk meningkatkan throughput dan customer experience
3. **Bundling & Upselling Strategy** untuk meningkatkan average transaction value

## Dampak yang Diharapkan

Jika semua rekomendasi diimplementasikan dengan baik:
- **Revenue tambahan**: $100,000-$200,000+ per tahun (scenario estimate)
- **Repeat rate**: Meningkat dari 3.04% ke 25%+ dalam 6 bulan
- **ATV**: Meningkat dari $6.85 ke $7.20-$7.50
- **Operational efficiency**: Peningkatan throughput 15-20% di peak hour

## Data Tambahan yang Masih Dibutuhkan

Untuk memperkuat rekomendasi dan memungkinkan analisis profitability:

1. **Cost data** - Biaya bahan baku, operasional, sewa, dan gaji per store
2. **Profit data** - Profit margin per produk dan per store
3. **Customer feedback** - Survey kepuasan pelanggan
4. **Competitor data** - Harga dan promosi kompetitor
5. **Foot traffic data** - Jumlah pengunjung potensial per lokasi
6. **Historical performance** - Data 2-3 tahun untuk trend analysis yang lebih robust

---

**Catatan Akhir:**

Semua rekomendasi dalam notebook ini **berbasis data aktual** dari analisis EDA dan dashboard. Angka-angka yang digunakan berasal langsung dari dataset `coffee_shop_sales_featured.csv` dan file ringkasan di `processed/eda/`. Tidak ada angka yang dibuat-buat atau diarang.

Dataset ini **tidak menyediakan data cost, profit, atau margin**, sehingga seluruh analisis difokuskan pada **revenue, transaction volume, quantity, dan ATV** sebagai proxy kinerja bisnis. Keputusan bisnis akhir harus mempertimbangkan data keuangan lengkap yang tersedia di sistem akuntansi perusahaan.